# Predict + grade `problem_ids_matched.csv` — transformers only (offline, no vLLM)

Loads the base **Nemotron-3-Nano** offline (kagglehub + Unsloth, same scripts as
`sft/nvidia-nemotron-v7-5.ipynb`), attaches the trained **LoRA adapter**, runs
**greedy** inference on every row, extracts `\boxed{}`, and grades per category
with the canonical verifier.

No vLLM (Kaggle has no internet). Generation is **per-prompt (batch=1)** — the
Nemotron Mamba-2 layers emit NaN logits on left-padded batches, so one prompt per
`generate` call (no padding) is the safe path; a NaN-logit guard is added as belt-
and-suspenders. Eval mirrors the competition: `temperature=0`, `max_tokens=7680`,
`max_model_len=8192`, `max_lora_rank=32`.

## 1. Config

In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

RUN_ON_KAGGLE = 1   # 1 = do the offline triton/ptxas/wheel installs (Kaggle GPU)

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# Trained LoRA adapter (dir with adapter_config.json + adapter_model.safetensors).
# Resolved via Kaggle MODEL slug -> dir -> auto-glob (see model-load cell).
ADAPTER_MODEL_SLUG = ""     # "" -> skip kagglehub; use ADAPTER_DIR
ADAPTER_DIR = os.environ.get(
    "ADAPTER_DIR",
    "/kaggle/input/models/ramkan07/nemotron-lora-adaptor/pytorch/default/1",
)

CSV_PATH = os.environ.get(
    "CSV_PATH",
    "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv",
)
RESULTS_CSV = os.environ.get("RESULTS_CSV", "/kaggle/working/matched_eval_results.csv")

# eval knobs (competition-locked)
MAX_MODEL_LEN = 8192
MAX_TOKENS    = 7680     # generation cap; per-prompt it's clamped to fit MAX_MODEL_LEN
TEMPERATURE   = 0.0      # greedy

# Match the SFT training format exactly (what the adapter learned).
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

LIMIT = None   # quick test: cap rows (None = all). Full run is slow (batch=1).
print("adapter:", ADAPTER_DIR, "| csv:", CSV_PATH)

## 2. Offline environment (triton wheel + ptxas + mamba/causal wheels) — from v7-5

In [ ]:
# Triton wheel (offline)
if RUN_ON_KAGGLE:
    import glob, subprocess, site
    candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
    print("Found Triton wheels:", candidates)
    if not candidates:
        raise FileNotFoundError("No Triton wheel found under /kaggle/input")
    target = "/kaggle/working/pydeps"; os.makedirs(target, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
                    "--upgrade", "--ignore-installed", candidates[0]], check=True)
    if target not in sys.path: sys.path.insert(0, target)
    site.addsitedir(target)
    import importlib.util
    print("triton spec:", importlib.util.find_spec("triton"))
else:
    print("RUN_ON_KAGGLE=0: skipping triton install.")

In [ ]:
# ptxas-blackwell shim (RTX 6000 Pro / Blackwell)
if RUN_ON_KAGGLE:
    import shutil, stat
    sys.path.insert(0, "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")
    ptxas_src = ("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/"
                 "triton/backends/nvidia/bin/ptxas-blackwell")
    ptxas_dst = "/tmp/ptxas-blackwell"
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src); dst_bin = "/tmp/triton_nvidia_bin"
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
        os.environ["TRITON_PTXAS_PATH"] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
    print("ptxas shim applied.")
else:
    print("RUN_ON_KAGGLE=0: skipping ptxas shim.")

In [ ]:
# Offline package install: unsloth/trl/peft/transformers + mamba_ssm + causal_conv1d
if RUN_ON_KAGGLE:
    import glob, subprocess
    def rwheels(p): return sorted(glob.glob(f"/kaggle/input/**/{p}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = rwheels("mamba_ssm-*.whl")
    all_causal = rwheels("causal*conv1d*.whl")
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("Needs a GPU runtime (Nemotron CUDA wheels).")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel dir not found: {packages_dir}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                    "--find-links", packages_dir, "peft", "transformers", "datasets", "accelerate", "bitsandbytes"], check=True)
    pick = lambda w: w[-1] if w else None
    cw, mw = pick(all_causal), pick(all_mamba)
    if cw: subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", cw], check=True)
    if mw: subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mw], check=True)
    else:  raise FileNotFoundError("No mamba_ssm wheel under /kaggle/input.")
    print("Offline packages installed.")
else:
    print("RUN_ON_KAGGLE=0: skipping offline package install.")

## 3. Load questions

In [ ]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
df = df[["id", "prompt", "answer", "type"]].copy()
df["answer"] = df["answer"].astype(str)
if LIMIT:
    df = df.head(LIMIT).reset_index(drop=True)
print(f"{len(df)} questions")
print(df["type"].value_counts())

## 4. Verifier — boxed extractor + per-category matcher (from nemo-v20)

In [ ]:
import re

_BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

def extract_boxed(text):
    """Brace-balanced \boxed{} parser (handles nested braces)."""
    idx = text.find("\\boxed{")
    if idx == -1:
        m = _BOXED_RE.search(text)
        return m.group(1).strip() if m else None
    depth, start = 1, idx + 7
    for i in range(start, len(text)):
        if text[i] == "{":   depth += 1
        elif text[i] == "}": depth -= 1
        if depth == 0:
            return text[start:i].strip()
    return text[start:].strip()

def _norm(s):
    return str(s).strip().lower().replace(" ", "").replace(",", "")

def _numeric_match(p, e, rel_tol=1e-2, abs_tol=1e-4):
    try:
        pf = float(str(p).replace(",", "").strip()); ef = float(str(e).replace(",", "").strip())
    except (ValueError, TypeError):
        return False
    return abs(pf - ef) <= max(rel_tol * max(1.0, abs(ef)), abs_tol)

def _integer_match(p, e, bases=(2, 8, 10, 16)):
    ps, es = _norm(p), _norm(e)
    def parse(s, b):
        try:
            if b == 16 and s.startswith("0x"): s = s[2:]
            elif b == 2 and s.startswith("0b"): s = s[2:]
            return int(s, b)
        except ValueError:
            return None
    for b1 in bases:
        pv = parse(ps, b1)
        if pv is None: continue
        for b2 in bases:
            ev = parse(es, b2)
            if ev is not None and pv == ev: return True
    return False

def verify_answer(predicted, expected, label):
    if predicted is None: return False
    pn, en = _norm(predicted), _norm(expected)
    if pn == en: return True
    if label in ("Gravitational Constant", "Unit Conversion"):
        return _numeric_match(predicted, expected)
    if label in ("Bit Manipulation", "Number Base Conversion"):
        return _integer_match(predicted, expected)
    if label == "Text Encryption":
        return pn.replace("'", "") == en.replace("'", "")
    if label == "Equation Transformation":
        return _numeric_match(predicted, expected) or pn == en
    return _numeric_match(predicted, expected)

# csv `type` -> verifier label (unmapped -> "Unknown": exact-norm + numeric fallback)
TYPE2LABEL = {
    "bit_manipulation":         "Bit Manipulation",
    "cipher":                   "Text Encryption",
    "unit_conversion":          "Unit Conversion",
    "gravity":                  "Gravitational Constant",
    "numeral":                  "Number Base Conversion",
    "equation_numeric_deduce":  "Equation Transformation",
    "equation_numeric_guess":   "Equation Transformation",
    "cryptarithm_deduce":       "Unknown",
    "cryptarithm_guess":        "Unknown",
}
print("verifier ready; types:", sorted(set(df["type"])))

## 5. Load base model (offline, Unsloth) + attach LoRA adapter

In [ ]:
import os, glob, torch
import kagglehub
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# resolve adapter dir: Kaggle MODEL slug -> ADAPTER_DIR -> auto-glob
def _resolve_adapter():
    if ADAPTER_MODEL_SLUG:
        try:
            p = kagglehub.model_download(ADAPTER_MODEL_SLUG)
            if os.path.exists(os.path.join(p, "adapter_config.json")): return p
            hits = glob.glob(os.path.join(p, "**", "adapter_config.json"), recursive=True)
            if hits: return os.path.dirname(sorted(hits, key=len)[0])
        except Exception as e:
            print("[adapter] kagglehub failed:", repr(e))
    if os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")): return ADAPTER_DIR
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    return os.path.dirname(sorted(hits, key=len)[0]) if hits else None

_adir = _resolve_adapter()
if _adir is None:
    raise FileNotFoundError(f"No adapter_config.json found (slug={ADAPTER_MODEL_SLUG!r}, dir={ADAPTER_DIR!r}).")
print("[adapter] using", _adir)

# Base model files are LOCAL via kagglehub (offline). Plain transformers +
# trust_remote_code loads the Nemotron-H custom modeling. NO Unsloth here: it pulls
# unsloth_zoo -> transformers.integrations.bitsandbytes ->
# is_bitsandbytes_multi_backend_available, a version-skew ImportError. Unsloth is a
# TRAINING optimizer; eval needs only transformers + the offline mamba/causal kernels.
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print("Model path:", MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
model = PeftModel.from_pretrained(model, _adir)   # attach trained LoRA
model.eval()
try: model.config.use_cache = True
except Exception: pass
print("base + adapter loaded (plain transformers, no Unsloth).")

## 6. Build prompts (eval format = training format)

In [ ]:
def build_prompt(problem: str) -> str:
    msgs = [{"role": "user", "content": str(problem) + PROMPT_SUFFIX}]
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=True, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

prompts = [build_prompt(p) for p in df["prompt"].tolist()]
print("example rendered prompt:\n", prompts[0][:600])

## 7. Generate — greedy, **per-prompt** (batch=1, Mamba-safe)

One prompt per `generate` call → no left-padding → no Mamba-2 NaN. `max_new_tokens`
is clamped per prompt so `prompt + generation <= MAX_MODEL_LEN`. A NaN-logit guard
catches any residual non-finite logits (must stay 0).

In [ ]:
import torch, time
from transformers import LogitsProcessor, LogitsProcessorList, StoppingCriteria, StoppingCriteriaList

# ── eval-speed cap: these answers need <~1.5k tokens; 7680 just makes it ramble ──
GEN_HARD_CAP = 2048      # max new tokens per prompt. raise only if 'truncated' is high.
LOG_EVERY    = 1         # print every N prompts (1 = every prompt)

class SanitizeLogits(LogitsProcessor):
    def __init__(self): self.hits = 0
    def __call__(self, input_ids, scores):
        if not torch.isfinite(scores).all():
            self.hits += 1
            scores = torch.nan_to_num(scores, nan=-1e4, posinf=1e4, neginf=-1e4)
        return scores

class StopOnBoxed(StoppingCriteria):
    """Stop the instant a COMPLETE \\boxed{...} appears -> don't generate the full cap."""
    def __init__(self, tokenizer, start_len):
        self.tok = tokenizer; self.start = start_len
    def __call__(self, input_ids, scores=None, **kw):
        gen = input_ids[0, self.start:]
        if gen.numel() < 4:
            return False
        tail = self.tok.decode(gen[-128:], skip_special_tokens=True)
        i = tail.rfind("\\boxed{")
        if i == -1:
            return False
        depth, j = 1, i + 7
        while j < len(tail) and depth > 0:
            if tail[j] == "{": depth += 1
            elif tail[j] == "}": depth -= 1
            j += 1
        return depth == 0

guard = SanitizeLogits(); lp = LogitsProcessorList([guard])
labels = df["type"].map(TYPE2LABEL).fillna("Unknown").tolist()
golds  = df["answer"].tolist()

gen_texts, truncated_flags, preds, oks = [], [], [], []
model.eval()
n_correct = 0
N = len(prompts)
print(f"=== Generating over {N} prompts (batch=1, stop-on-boxed, cap={GEN_HARD_CAP}) ===", flush=True)
t_start = time.time()

for i, p in enumerate(prompts):
    enc = tokenizer(p, return_tensors="pt", truncation=True,
                    max_length=MAX_MODEL_LEN - 256).to(model.device)
    plen = enc["input_ids"].shape[1]
    mnt = max(16, min(GEN_HARD_CAP, MAX_MODEL_LEN - plen - 8))
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=mnt,
            do_sample=False, temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            logits_processor=lp,
            stopping_criteria=StoppingCriteriaList([StopOnBoxed(tokenizer, plen)]),
        )
    dt = time.time() - t0
    new = out[0][plen:]
    glen = int(new.shape[0])
    txt = tokenizer.decode(new, skip_special_tokens=True)
    trunc = glen >= mnt
    pred = None if trunc else extract_boxed(txt)
    ok = verify_answer(pred, golds[i], labels[i])
    n_correct += int(ok)
    gen_texts.append(txt); truncated_flags.append(trunc); preds.append(pred); oks.append(ok)

    # ── GUARANTEED per-prompt log (plain print + flush) ──
    if i % LOG_EVERY == 0 or i == N - 1:
        print(f"[{i+1:>4}/{N}] {dt:4.1f}s len={glen:<4} {labels[i][:16]:<16} "
              f"pred={str(pred)[:22]!r:<24} gold={str(golds[i])[:18]!r:<20} "
              f"{'OK ' if ok else ' X '} run_acc={n_correct/(i+1)*100:5.1f}%", flush=True)

df["generation"] = gen_texts
df["truncated"]  = truncated_flags
df["pred"]       = preds
df["correct_run"]= oks

print(f"\n=== DONE in {(time.time()-t_start)/60:.1f} min ===", flush=True)
print(f"NaN-guard hits : {guard.hits} (must be 0)")
print(f"running accuracy: {n_correct}/{N} = {n_correct/N*100:.2f}%")
print(f"truncated rate : {df['truncated'].mean()*100:.2f}%")
print(f"no-boxed rate  : {df['pred'].isna().mean()*100:.2f}%")

## 8. Grade — per-category accuracy

In [ ]:
df["label"]   = df["type"].map(TYPE2LABEL).fillna("Unknown")
df["correct"] = [verify_answer(p, e, lab)
                 for p, e, lab in zip(df["pred"], df["answer"], df["label"])]

report = (df.groupby("type")
            .agg(total=("correct", "count"), passed=("correct", "sum"),
                 truncated=("truncated", "sum"))
            .assign(failed=lambda x: x["total"] - x["passed"]))
report["acc_%"]   = (100 * report["passed"]    / report["total"]).round(2)
report["trunc_%"] = (100 * report["truncated"] / report["total"]).round(2)
report = report[["total", "passed", "failed", "truncated", "acc_%", "trunc_%"]].sort_values("acc_%")

print(report.to_string())
print(f"\nOVERALL  : {df['correct'].sum()}/{len(df)} = {df['correct'].mean()*100:.2f}%")
print(f"TRUNCATED: {df['truncated'].sum()}/{len(df)} = {df['truncated'].mean()*100:.2f}%")

df.to_csv(RESULTS_CSV, index=False)
print("per-row results ->", RESULTS_CSV)

## 9. Inspect failures (optional)

In [ ]:
fails = df[~df["correct"]][["id", "type", "answer", "pred", "truncated"]]
print(f"{len(fails)} failures")
fails.head(20)